In [ ]:
# Update package list and install Python 3.11
!sudo apt-get update -y
!sudo apt-get install python3.11 python3.11-venv python3.11-dev -y

# Create a virtual environment
!python3.11 -m venv xtts_env

# Install TTS, PyTorch, and a compatible numpy version INTO the virtual environment
!xtts_env/bin/pip install -q TTS torch torchaudio "numpy<2.0"

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,703 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,312 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,607 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/

In [ ]:
!wget -q -O male_voice.wav https://huggingface.co/spaces/coqui/xtts/resolve/main/examples/male.wav

In [ ]:
!xtts_env/bin/pip install -q "transformers<4.50.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 110.7 MB/s eta 0:00:00


In [ ]:
!xtts_env/bin/pip install -q torchcodec soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.9 MB/s eta 0:00:00


In [ ]:
%%writefile run_xtts.py
import os
import torch

# --- PyTorch 2.6+ Compatibility Patch ---
# Force torch.load to allow legacy Coqui config objects safely
orig_load = torch.load
def patched_load(*args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return orig_load(*args, **kwargs)
torch.load = patched_load
# ----------------------------------------

from TTS.api import TTS

# Automatically agree to the Coqui Public Model License
os.environ["COQUI_TOS_AGREED"] = "1"

# Confirm GPU is being used
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {device}...")

# Initialize XTTS v2
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

# Set up text and file paths
hindi_text = "नमस्ते, मेरा नाम पलक है।"
reference_audio = "my_voice.mp3.acc"
output_path = "hindi_output.wav"

print("Generating Hindi speech...")

# Generate the speech
tts.tts_to_file(
    text=hindi_text,
    speaker_wav=reference_audio,
    language="hi",
    file_path=output_path
)

print("Audio saved successfully!")

Overwriting run_xtts.py


In [ ]:
import os

if os.path.exists("my_voice.mp3.acc"):
    print("✅ Perfect! 'my_voice.wav' is ready and waiting in your folder.")
else:
    print("❌ Hmm, 'my_voice.wav' wasn't found. Go back and run the recording cell again!")

❌ Hmm, 'my_voice.wav' wasn't found. Go back and run the recording cell again!


In [ ]:
# Force a safe matplotlib backend and run the script
!MPLBACKEND=Agg xtts_env/bin/python run_xtts.py

# Display the audio player
import os
from IPython.display import Audio, display

if os.path.exists("hindi_output.wav"):
    print("\n Success! Listen to your generated Hindi speech below:")
    display(Audio("hindi_output.wav", autoplay=True))
else:
    print("\nCheck the error log above!")

Loading model on cuda...
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts
GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Generating Hindi speech...
 > Text splitted

In [ ]:
from google.colab import files
import os

print("Select your recorded voice clip from your computer:")
uploaded = files.upload()

# Automatically rename whatever file you upload to 'my_voice.wav'
for filename in uploaded.keys():
    os.rename(filename, 'my_voice.wav')
    print("✅ Successfully uploaded and configured 'my_voice.wav'!")

Select your recorded voice clip from your computer:


Saving my_voice.mp3.aac to my_voice.mp3 (1).aac
✅ Successfully uploaded and configured 'my_voice.wav'!
